# 49 · v3: k = 4 sem km por treino e sem plano

Mesma população (13.999 assinantes ativos que treinaram nos 90 dias até 30/09/2026), mesma semente e
mesmo preparo dos notebooks 44 a 48. A **v3** é a v1 sem canal menos duas variáveis:

- **km por treino** sai: o 0 atribuído a quem só treina indoor formou um grupo sozinho no k = 5 (notebook 48);
- **plano** sai: separava pouco (R² 0,08 no k = 4).

Ficam cinco variáveis, todas em z-score: treinos por semana, dias sem treino (log), tempo de casa (log),
% dos treinos ao ar livre e engajamento (ordinal). Km e plano continuam no perfil dos grupos.
Comparação com o k = 4 da v1 sem canal (notebook 47). Ordem de execução: 39 → 41 → 42 → 43 → 44 → 45 → 46 → 47 → 48 → 49.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
KS = [4]; SEED = 0
AR_LIVRE = ["corrida", "caminhada", "pedal"]

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master:", m.shape)
print(m.segmento_regra.value_counts().to_string())
e = m[m.segmento_regra == "elegível ao k-means"].copy().reset_index(drop=True)
assert (e.data_referencia == CORTE).all() and e.ativa_no_corte.all()
print("\nelegíveis ao k-means:", len(e), "| contas distintas:", e.user_id.nunique())

tabela master: (63929, 119)
segmento_regra
fora do escopo (não é assinante ativo)    49176
elegível ao k-means                       13999
inativo em 90 dias                          754

elegíveis ao k-means: 13999 | contas distintas: 13999


## 1. % dos treinos ao ar livre

In [2]:
e["pct_ar_livre"] = (e[["treinos_" + t for t in AR_LIVRE]].sum(axis=1) / e.treinos_90d * 100).astype(float)
print("treinos_90d > 0 em todos os elegíveis:", bool((e.treinos_90d > 0).all()))
print(e.pct_ar_livre.describe().round(1).to_dict())
print("0% ao ar livre:", int((e.pct_ar_livre == 0).sum()), "| 100% ao ar livre:", int((e.pct_ar_livre == 100).sum()))

treinos_90d > 0 em todos os elegíveis: True
{'count': 13999.0, 'mean': 54.4, 'std': 34.6, 'min': 0.0, '25%': 16.7, '50%': 71.4, '75%': 83.3, 'max': 100.0}
0% ao ar livre: 691 | 100% ao ar livre: 1087


## 2. Km por treino

Média de `distancia_km` dos treinos ao ar livre (corrida, caminhada, pedal) na janela de 90 dias até o
corte (mesma janela da tabela master: 0 a 89 dias antes de 30/09/2026), só treinos com GPS válido e
distância maior que zero. Quem não tem nenhum treino com distância na janela fica com **0 km**.

In [3]:
a = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "distancia_km", "gps_ok"])
print("treinos no arquivo:", len(a))
a = a[a.user_id.isin(e.user_id)]
print("-> dos elegíveis:", len(a))
a = a[(CORTE - a.data).dt.days.between(0, 89)]
print("-> na janela de 90 dias:", len(a))
al = a[a.tipo.isin(AR_LIVRE)]
print("-> ao ar livre:", len(al), "| GPS falhou:", int((~al.gps_ok).sum()), "| GPS ok sem distância ou zero:", int((al.gps_ok & ~(al.distancia_km > 0)).sum()))
d = al[al.gps_ok & (al.distancia_km > 0)]
print("-> com distância válida:", len(d), "| contas:", d.user_id.nunique())
km = d.groupby("user_id").distancia_km.mean().rename("km_por_treino")
e = e.merge(km, on="user_id", how="left")
fal = set(al.user_id) - set(d.user_id)
print("\ncontas sem treino com distância (ficam com 0 km):", int(e.km_por_treino.isna().sum()))
print("  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou):", len(fal))
e["km_por_treino"] = e.km_por_treino.fillna(0)
e["km_por_treino_log"] = np.log1p(e.km_por_treino)
print("\nkm por treino, quem tem distância:", e.km_por_treino[e.km_por_treino > 0].describe().round(1).to_dict())
print("km por treino por tipo principal (mediana, quem tem distância):")
print(e[e.km_por_treino > 0].groupby("tipo_principal").km_por_treino.median().round(1).to_string())
print("assimetria antes e depois do log:", round(e.km_por_treino.skew(), 2), "->", round(e.km_por_treino_log.skew(), 2))

treinos no arquivo: 1252488


-> dos elegíveis: 941195


-> na janela de 90 dias: 372012
-> ao ar livre: 199229 | GPS falhou: 0 | GPS ok sem distância ou zero: 0
-> com distância válida: 199229 | contas: 13308

contas sem treino com distância (ficam com 0 km): 691
  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou): 0

km por treino, quem tem distância: {'count': 13308.0, 'mean': 8.7, 'std': 9.5, 'min': 1.9, '25%': 5.0, '50%': 5.8, '75%': 7.6, 'max': 162.8}
km por treino por tipo principal (mediana, quem tem distância):
tipo_principal
caminhada    5.40
corrida      7.00
forca        5.20
funcional    5.20
pedal       27.30
yoga         5.10
assimetria antes e depois do log: 6.28 -> -0.17


## 3. Engajamento, plano e canal

In [4]:
ORD = {"possível churn": 0, "baixo": 1, "médio": 2, "alto": 3}
e["engajamento_ord"] = e.nivel_engajamento.map(ORD)
assert e.engajamento_ord.notna().all()
print(e.nivel_engajamento.value_counts().to_string())
e["plano_anual"] = (e.plano == "anual").astype(int)
print("\nplano anual:", int(e.plano_anual.sum()), f"({e.plano_anual.mean():.1%})")
canal = pd.get_dummies(e.canal_aquisicao, prefix="canal").astype(int)
e = pd.concat([e, canal], axis=1)
CANAIS = list(canal.columns)
print("\ncanal:"); print(e.canal_aquisicao.value_counts().to_string())

nivel_engajamento
alto              10689
médio              2366
possível churn      579
baixo               365

plano anual: 7727 (55.2%)

canal:
canal_aquisicao
organico                4410
influenciador           2379
meta_ads                2294
indicacao               2113
google_ads              1435
parceria_corporativa    1368


## 4. Matriz

In [5]:
NUM = ["treinos_por_semana", "dias_sem_treinar_log", "duracao_dias_log", "pct_ar_livre", "engajamento_ord"]
IND = []
sc = StandardScaler().fit(e[NUM])
X = pd.DataFrame(sc.transform(e[NUM]), columns=[c + "_z" for c in NUM])
print("matriz:", X.shape, "| vazios:", int(X.isna().sum().sum()))

matriz: (13999, 5) | vazios: 0


## 5. k = 4: as lentes de avaliação

Grupos numerados por tamanho (1 = maior). Silhueta com todos os pontos. Estabilidade: 5 sementes diferentes
e 5 reamostragens com reposição, pelo índice de Rand ajustado (a aula pede acima de 0,9).

In [6]:
def r2(col, lab):
    tot = ((col - col.mean()) ** 2).sum()
    return sum(len(s) * (s.mean() - col.mean()) ** 2 for _, s in col.groupby(lab)) / tot

res, rot, sep = [], {}, {}
for k in KS:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    lab = km.labels_
    mapa = {g: i + 1 for i, g in enumerate(pd.Series(lab).value_counts().index)}
    rot[k] = pd.Series(lab).map(mapa).values
    seeds = [adjusted_rand_score(lab, KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(X)) for s in [1, 2, 3, 4, 5]]
    rng = np.random.default_rng(SEED); boot = []
    for i in range(5):
        idx = rng.integers(0, len(X), len(X))
        boot.append(adjusted_rand_score(lab, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X.iloc[idx]).predict(X)))
    sep[f"k={k}"] = pd.Series({c: r2(X[c], lab) for c in X.columns})
    res.append({"k": k, "inercia": km.inertia_, "silhueta": silhouette_score(X, lab),
                "rand_sementes_min": min(seeds), "rand_reamostra_media": np.mean(boot), "rand_reamostra_min": min(boot),
                "menor_grupo": int(np.bincount(lab).min()), "tamanhos": sorted(np.bincount(lab).tolist(), reverse=True)})
res = pd.DataFrame(res).set_index("k")
res["ganho_do_grupo_seguinte"] = -res.inercia.diff().shift(-1)
print(res.round(3).to_string())
print("\nquanto cada variável separa os grupos (R² entre grupos):")
print(pd.DataFrame(sep).round(3).sort_values("k=4", ascending=False).to_string())

    inercia  silhueta  rand_sementes_min  rand_reamostra_media  rand_reamostra_min  menor_grupo                  tamanhos  ganho_do_grupo_seguinte
k                                                                                                                                                 
4 31,118.77      0.30               1.00                  0.99                0.98         2092  [5001, 4030, 2876, 2092]                      NaN

quanto cada variável separa os grupos (R² entre grupos):
                        k=4
engajamento_ord_z      0.69
dias_sem_treinar_log_z 0.62
pct_ar_livre_z         0.60
duracao_dias_log_z     0.53
treinos_por_semana_z   0.34


## 6. Comparação com o k = 4 da v1 sem canal (notebook 47)

Linhas: grupo da v1 sem canal; colunas: grupo da v3. Os dois numerados por tamanho.

In [7]:
ref = pd.read_parquet(P / "kmeans_v1sc_k234_rotulos.parquet", columns=["user_id", "grupo_k4"])
c = e[["user_id"]].assign(v3=rot[4]).merge(ref, on="user_id", how="left")
assert c.grupo_k4.notna().all()
print("Rand ajustado, v1 sem canal x v3 (k = 4):", round(adjusted_rand_score(c.grupo_k4, c.v3), 3))
print(pd.crosstab(c.grupo_k4, c.v3, rownames=["v1 sem canal"], colnames=["v3"], margins=True))
print("\nperfil da v1 sem canal, k = 4 (notebook 47), para comparar:")
print(pd.read_csv(S / "kmeans_v1sc_k234_perfis.csv").query("k == 4").set_index("grupo").drop(columns="k").round(1).T)

Rand ajustado, v1 sem canal x v3 (k = 4): 0.931
v3               1     2     3     4    All
v1 sem canal                               
1             4955     6    41    96   5098
2                2  4016   136    36   4190
3                2     0  2675     7   2684
4               42     8    24  1953   2027
All           5001  4030  2876  2092  13999

perfil da v1 sem canal, k = 4 (notebook 47), para comparar:
grupo                            1          2        3        4
assinantes                    5098       4190     2684     2027
pct_base                     36.40      29.90    19.20    14.50
treinos_por_semana            2.60       2.30     0.50     0.90
dias_sem_treinar              1.00       1.00    20.00     2.00
meses_de_casa                10.70      10.30    10.00     0.90
pct_ar_livre                 80.80      14.30    77.80    70.00
km_por_treino                 7.10       5.20     6.00     5.70
pct_anual                    53.70      50.40    80.70    35.20
pct_alt

## 7. Perfil dos grupos em cada k

Medianas na unidade original e a lente de três fatores: ativos (tamanho), engajamento (níveis) e sinal de
saída (renovação automática desligada no corte; não há saída observada depois de 30/09/2026).

In [8]:
def perfil(gr):
    g = e.assign(grupo=gr).groupby("grupo")
    return pd.DataFrame({
        "assinantes": g.size(), "pct_base": g.size() / len(e) * 100,
        "treinos_por_semana": g.treinos_por_semana.median(), "dias_sem_treinar": g.dias_sem_treinar.median(),
        "meses_de_casa": g.meses_de_casa.median(), "pct_ar_livre": g.pct_ar_livre.median(),
        "km_por_treino": g.km_por_treino.median(), "pct_anual": g.plano_anual.mean() * 100,
        "pct_alto": g.nivel_engajamento.apply(lambda s: (s == "alto").mean() * 100),
        "pct_medio": g.nivel_engajamento.apply(lambda s: (s == "médio").mean() * 100),
        "pct_baixo": g.nivel_engajamento.apply(lambda s: (s == "baixo").mean() * 100),
        "pct_possivel_churn": g.nivel_engajamento.apply(lambda s: (s == "possível churn").mean() * 100),
        "pct_renovacao_desligada": g.renovacao_desligada.apply(lambda s: s.astype(float).mean() * 100),
        "tipo_principal_mais_comum": g.tipo_principal.agg(lambda s: s.value_counts().index[0]),
        "pct_influenciador": g.canal_aquisicao.apply(lambda s: (s == "influenciador").mean() * 100),
    })
PERF = {}
for k in KS:
    PERF[k] = perfil(rot[k]); print(f"===== k = {k}"); print(PERF[k].round(1).T); print()
print("base: renovação desligada", round(e.renovacao_desligada.astype(float).mean() * 100, 1), "% | engajamento alto",
      round((e.nivel_engajamento == "alto").mean() * 100, 1), "% | anual", round(e.plano_anual.mean() * 100, 1), "%")

===== k = 4
grupo                            1          2        3        4
assinantes                    5001       4030     2876     2092
pct_base                     35.70      28.80    20.50    14.90
treinos_por_semana            2.60       2.40     0.60     0.90
dias_sem_treinar              1.00       1.00    18.00     2.00
meses_de_casa                10.80      10.50    10.00     0.90
pct_ar_livre                 80.80      14.50    76.30    70.00
km_por_treino                 7.00       5.20     5.90     5.80
pct_anual                    53.50      50.00    78.10    37.60
pct_alto                     98.80      95.00     0.00    91.90
pct_medio                     1.20       5.00    67.20     8.10
pct_baixo                     0.00       0.00    12.70     0.00
pct_possivel_churn            0.00       0.00    20.10     0.00
pct_renovacao_desligada       4.70       4.40    52.20     3.40
tipo_principal_mais_comum  corrida  funcional  corrida  corrida
pct_influenciador           

## 8. Salvar

In [9]:
out = e[["user_id"]].assign(**{f"grupo_k{k}": rot[k] for k in KS})
out.to_parquet(P / "kmeans_v3_k4_rotulos.parquet", index=False)
res.assign(tamanhos=res.tamanhos.astype(str)).round(4).to_csv(S / "kmeans_v3_k4_metricas.csv", encoding="utf-8-sig")
pd.concat(PERF, names=["k", "grupo"]).round(2).to_csv(S / "kmeans_v3_k4_perfis.csv", encoding="utf-8-sig")
pd.DataFrame(sep).round(4).to_csv(S / "kmeans_v3_k4_separacao_por_variavel.csv", encoding="utf-8-sig", index_label="variavel")
print("rótulos:", out.shape, "-> dados/processados/kmeans_v3_k4_rotulos.parquet")
print("saídas: kmeans_v3_k4_metricas.csv, _perfis.csv, _separacao_por_variavel.csv")

rótulos: (13999, 2) -> dados/processados/kmeans_v3_k4_rotulos.parquet
saídas: kmeans_v3_k4_metricas.csv, _perfis.csv, _separacao_por_variavel.csv
